# History
- AutoEncoder, AE：自编码器。训练过程是自监督的。
> input: $x$\
> encoder: $z = e(x)$\
> decoder: $\hat x = d(z)$\
> objective: $z = \arg\min ||x - \hat x ||^2$
>
> 问题：过拟合。
- 路线一 —— 解决AE的过拟合问题：
    - Variational Autoencoder, VAE：变分自编码器。
    > **改动1**：编码器的输出不再是一个确定的数，而是输出一个均值和方差，然后从正态分布中随机采样，作为解码器的输入，以此增加随机扰动；
    >
    > **改动2**：添加一个学习目标，让编码器的输出和标准正态分布尽可能相似；
    >
    > loss: $||\hat x - x ||^2 - sim(N(\mu, \sigma^2), N(0, I))$
    >
    > 分布与分布之间的误差用KL散度表示；
    >
    > 问题：VAE的正则化目标是均方误差尽可能小，容易导致结果平滑，生成图像模糊。
    - Denoising Diffusion Probabilistic Model, DDPM：去噪扩散概率模型。
    > **改动1**：编码器是一系列不可学习（固定）的加噪声操作；
    >
    > **改动2**：编码器是一系列可学习的去噪声过程；
    >
    > 图像尺寸始终不变；
    >
    > 问题：DDPM虽然效果很好，但是运行速度一般，比较耗资源。
- 路线二 —— 用AE压缩图像间接实现图像生成：
    - Vector Quantised-Variational AutoEncoder, VQ-VAE：向量离散化-变分自编码。
    > 将AE中encoder输出的连续向量离散化，实现将图像压缩成离散向量，即将图像等比例压缩成离散的小图像。
    >
    > 两阶段图像生成流程，因为1.transformer等生成模型只支持生成离散图像，需要另一个模型把连续的颜色值变成离散值以兼容；2.减少模型的运算量.
    >
    > **训练时**，先训练一个图像压缩模型(VQVAE)，再训练一个生成压缩图像的模型(比如transformer)。第一阶段训练完后，模型encoder知道如何压缩，decoder知道如何解压，此时得到一个高质量latent空间；第二阶段训练时encoder固定，把所有图片变成压缩后的离散小图像，训练生成图像的模型；
    >
    > **生成时**，先用第二个模型生成出一个压缩图像，再用第一个模型的解码器把压缩图像复原成真实图像；
    >
    > 缓解过拟合：由于经过向量离散化后，此时解码器的输入不再是编码器的输出，而是嵌入层中另一个模型的输出向量

# Stable Diffusion

- 论文提出 **隐扩散模型（latent diffusion model, LDM）**：
    - 核心解决：减少像素空间扩散模型的运算开销
    - LDM 借助 VQ-VAE「先压缩、再生成」的两阶段图像生成思想，把扩散模型用于 AE 的隐空间上，在几乎不降低生成质量的前提下减少了计算量 $\Rightarrow$ *LDM的AE怎么设计以达到压缩比例与质量的平衡*
    - 另外还支持带约束图像合成及纯卷积图像超分辨率 $\Rightarrow$ *LDM怎么用交叉注意力机制实现带约束的图像合成*
    - 学习目标：原理和 DDPM 完全一致，只不过把训练图片从像素空间上的真实图片 $x_0$ 变成了隐空间上的压缩图片 $z_0$，每轮的带噪图片由 $x_t$ 变成了隐空间上的带噪图片 $z_t$。在训练时，相比DDPM，只需要多对 $x_0$ 用一次编码器变成 $z_0$ 即可。
        - DDPM objective：$$LOSS_{DM} = \mathbb{E}_{x,\epsilon \sim \mathcal{N}(0, 1), t}\parallel \epsilon - \epsilon_{\theta}(x_t, t)\parallel^2_2 $$
        - LDM objective：$$LOSS_{LDM} = \mathbb{E}_{encoder(x),\epsilon \sim \mathcal{N}(0, 1), t}\parallel \epsilon - \epsilon_{\theta}(z_t, t)\parallel^2_2 $$
    - 约束机制：
        - 拼接：把额外的信息和扩散模型原本的输入 $z_t$ 拼接起来 $\Rightarrow$ *适合有空间信息约束，如语义分割图*
        - 交叉注意力：K, V 换成来自约束的信息，交叉注意力相当于约束信息和原本输入信息进行了一次信息融合 $\Rightarrow$ *适合作用于全局的约束，如文本描述*
    - 实验效果：
        - 感知压缩程度的折中：f一般取8。
            - 训练速度上，下采样比例 f 过小时，扩散模型把过多精力放在本应由压缩模型负责的感知压缩上；采样比例过大时，图像信息在压缩中损失过多。综合来看，下采样比例 f 在4-16比较好。
            - DDIM采样步数越少，采样速度越快，生成图像质量越差。无论采样步数多与少，f 太小或太大的效果都不行，整体还是 f 在4-8比较好。
        - 图像生成效果：表示采样质量的FID和表示数据分布覆盖率的精确率和召回率
            - precision：分类中表示所有被分为正的样本中有多少是分对了的。在此为，生成模型生成的样本落在真实分布的比例，描述采样质量；
            - recall：分类中表示所有真值为正的样本中有多少被成功分类为正的。在此为，真实分布的样本落在生成模型的分布的比例，描述生成分布与真实分布的覆盖情况。
            - LDM表现整体还行，虽然FID无法超越GAN等扩散模型，但是precision和recall有一定优势，且所用参数少
        - 文生图能力
    - 不足：
        - 尽管计算需求比其他像素空间上的扩散模型要少得多，但受制于扩散模型本身的串行采样，它的采样速度还是比GAN慢。
        - LDM使用了一个自编码器来压缩图像，重建图像带来的精度损失会成为某些精准像素值的任务的性能瓶颈。

## LDM采样算法 —— 伪代码：

- 最早的DDPM算法：
    - DDPMScheduler()类，专门维护扩散模型的 $\alpha, \beta$ 变量；
    - U-Net神经完了unet()，用于计算去噪过程中图像应该去除的噪声 eps；
    - 用 randn() 从标准正态分布中采样一个纯噪声图像 $x_t$。将其逐渐去噪，最终变成一副图片。去噪过程中，时刻 t 会从总时刻 T 遍历至1。每一轮去噪，UNet根据这一时刻的图像 $x_t$ 和当前时间戳 t 估计出此时刻应去除的噪声 eps，然后就知道下一步图像的均值，再从DDPM调度类中直接获取方差（方差与$x_t$和t无关，是一个常量）。于是根据DDPM公式，知道均值和方差后就能采样出下一步的图像。反复执行去噪循环，$x_t$ 会从纯噪声图像变成一副有意义的图像。
```
def ddpm_sample(image_shape):
  ddpm_scheduler = DDPMScheduler()
  unet = UNet()
  xt = randn(image_shape)
  T = 1000
  for t in T ... 1:
    eps = unet(xt, t)
    std = ddpm_scheduler.get_std(t)
    xt = ddpm_scheduler.get_xt_prev(xt, t, eps, std)
  return xt
```

- DDIM对DDPM进行了两点改进：1）去噪的有效步数可以少于 T 步，由另一个变量 ddim_steps 决定；2）采样的方差大小可以由 eta 决定。
    - ddim_steps 是去噪训练的执行次数，DDIM调度器可以根据 ddim_steps 生成所有被使用到的 t。
    - eta 用来计算方差，一般被设为0，如果设为1则退化为DDPM。
```
def ddim_sample(image_shape, ddim_steps = 20, eta = 0):
  ddim_scheduler = DDIMScheduler()
  unet = UNet()
  xt = randn(image_shape)
  T = 1000
  timesteps = ddim_scheduler.get_timesteps(T, ddim_steps) # [1000, 950, 900, ...]
  for t in timesteps:
    eps = unet(xt, t)
    std = ddim_scheduler.get_std(t, eta)
    xt = ddim_scheduler.get_xt_prev(xt, t, eps, std)
  return xt
```

- 在DDIM的基础上，LDM从生成像素空间上的图像变成了生成压缩空间上的图像。需要再做一次解码才能变回真实图像。即需要多准备一个VAE，并对最后的隐空间图像 $z_t$ 解码:
```
def ldm_ddim_sample(image_shape, ddim_steps = 20, eta = 0):
  ddim_scheduler = DDIMScheduler()
  vae = VAE()
  unet = UNet()
  zt = randn(image_shape)
  T = 1000
  timesteps = ddim_scheduler.get_timesteps(T, ddim_steps) # [1000, 950, 900, ...]
  for t in timesteps:
    eps = unet(zt, t)
    std = ddim_scheduler.get_std(t, eta)
    zt = ddim_scheduler.get_xt_prev(zt, t, eps, std)
  xt = vae.decoder.decode(zt)
  return xt
```

- 实现文生图，需要额外的文本输入 text。文本编码器将文本编码成张量 c，输入进unet。
```
def ldm_text_to_image(image_shape, text, ddim_steps = 20, eta = 0):
  ddim_scheduler = DDIMScheduler()
  vae = VAE()
  unet = UNet()
  zt = randn(image_shape)
  T = 1000
  timesteps = ddim_scheduler.get_timesteps(T, ddim_steps) # [1000, 950, 900, ...]

  text_encoder = CLIP()
  c = text_encoder.encode(text)

  for t = timesteps:
    eps = unet(zt, t, c)
    std = ddim_scheduler.get_std(t, eta)
    zt = ddim_scheduler.get_xt_prev(zt, t, eps, std)
  xt = vae.decoder.decode(zt)
  return xt
```

## U-Net 结构组成

- 最早的 U-Net：
    - 整体上看，U-Net由若干个大层组成。特征在每一大层会被下采样成尺寸更小的特征，再被上采样回原尺寸的特征。整个网络构成一个U形结构。
    - 下采样后，特征的通道数会变多。一般情况下，每次下采样后图像尺寸减半，通道数翻倍。上采样过程则反之。
    - 为了防止信息在下采样的过程中丢失，U-Net每一大层在下采样前的输出会作为额外输入拼接到每一大层上采样前的输入上。这种数据连接方式类似于ResNet中的「短路连接」。
- DDPM使用的改进版 U-Net：
    - 原来的卷积层被替换成了ResNet中的**残差卷积模块**。每一大层有若干个这样的子模块。对于较深的大层，残差卷积模块后面还会接一个**自注意力模块**。
    - 原来模型每一大层只有一个短路连接。现在每个大层下采样部分的每个子模块的输出都会额外输入到其对称的上采样部分的子模块上。直观上来看，就是短路连接更多了一点，输入信息更不容易在下采样过程中丢失。
- 给 U-Net 添加约束信息的方法：把自注意力模块换成**交叉注意力模块**
    - 把DDPM中U-Net的自注意力模块换成了标准的Transformer模块，约束信息 C 可以作为 CrossAttention的 K, V 输入进模块中。

In [1]:
from diffusers import DiffusionPipeline, StableDiffusionPipeline
import torch

pipeline = DiffusionPipeline.from_pretrained(
    "sd-legacy/stable-diffusion-v1-5",
    cache_dir="D:/agent/diffusion model/model",
    torch_dtype = torch.float16,
).to("cuda")
pipeline("A cat in a pot").images[0].save("output.jpg")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: D:\agent\diffusion model\model\models--sd-legacy--stable-diffusion-v1-5\snapshots\451f4fe16113bff5a5d2269ed5ad43b0592e9a14\safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: D:\agent\diffusion model\model\models--sd-legacy--stable-diffusion-v1-5\snapshots\451f4fe16113bff5a5d2269ed5ad43b0592e9a14\text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/50 [00:00<?, ?it/s]

# LoRA in Stable Diffusion

## 原理：Low-Rank Adaptation(低秩适配)

模型在微调过程中，对于参数 $W \in \mathbb{R}^{d \times d}$，实际上应该维护的是其变化量 $\Delta W \in \mathbb{R}^{d \times d}$，训练时的参数用 $W + \Delta W$ 表示。此时，微调需要学习的变化量参数矩阵 $\Delta W$ 和原始参数矩阵 $W$ 是一样大的，这样并不高效。由于这个变化量矩阵蕴含的信息并没有那么多，是一个高度稀疏的低秩矩阵，故将其拆分为两个低秩矩阵的乘积：
$$\Delta W = BA$$
其中，$A \in \mathbb{R}^{r \times d}$，$B \in \mathbb{R}^{d \times r}$，r 是一个比 d 小得多的数（一般令r=4，8，16即可）。

在SD中使用LoRA，一般会对SD的 U-Net 的所有多头注意力模块的所有参数矩阵做微调，即对多头注意力模块的四个矩阵 $W_Q, W_K, W_V, W_{out}$ 进行微调。

三种应用：
- **还原单幅图像**：为编辑给定的图片，先学会生成一模一样的图片，再在此基础上进行修改。
    - 只用这一张图片来微调SD，让SD在这张图片上过拟合，输出就和这种图片非常相似了
- **风格调整**：希望SD只生产某一画风，或某一人物的图片，只需要在符合要求的训练集上训练SD LoRA即可
- **训练目标调整**：LoRA最初是把一个预训练模型适配到另一任务上，修改U-Net的训练目标，以提升SD的能力